In [1]:
import pandas as pd
import xgboost as xgb
import numpy as np
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from sklearn.metrics import f1_score

In [2]:
# Load the dataset
df = pd.read_csv(r'C:\Users\khiew\Downloads\FYP Reduced Dataset.csv')

In [3]:
# Drop diseases with less than 1000 instances
disease_counts = df['diseases'].value_counts()
valid_diseases = disease_counts[disease_counts >= 500].index
df = df[df['diseases'].isin(valid_diseases)]

# Assuming that the target variable is 'diseases' and all other variables are input features
X = df.drop('diseases', axis=1)
y = df['diseases']

# Encode the target variable (diseases) if it's a categorical variable
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)
print("Number of remaining classes in training set:", len(np.unique(y_train)))
print("Number of rows left:", len(df))

Number of remaining classes in training set: 201
Number of rows left: 168499


In [4]:
# Apply SMOTE for class balancing in the training set
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# Print the new class distribution after SMOTE
print("Class distribution after SMOTE:")
print(pd.Series(y_train_resampled).value_counts())
print("Number of remaining classes in training set:", len(np.unique(y_train_resampled)))
# Print the number of rows in the resampled training set
print("Number of rows in the resampled training set:", len(X_train_resampled))

Class distribution after SMOTE:
88     1002
150    1002
123    1002
125    1002
141    1002
       ... 
108    1002
39     1002
96     1002
192    1002
118    1002
Name: count, Length: 201, dtype: int64
Number of remaining classes in training set: 201
Number of rows in the resampled training set: 201402


In [5]:
def objective(trial):
    params = {
        'objective': 'multi:softprob',
        'num_class': len(np.unique(y_train)),
        'tree_method': 'hist',
        'eval_metric': 'mlogloss',
        
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 5),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 5),
        
        # Try including early stopping directly in the model parameters
        'early_stopping_rounds': 50,
    }
    
    # Create XGBoost model with parameters from Optuna
    model = xgb.XGBClassifier(**params)
    
    # In XGBoost 3.0.0, try a simpler fit call
    model.fit(
        X_train_resampled,
        y_train_resampled,
        eval_set=[(X_test, y_test)],
        verbose=False
    )
    
    preds = model.predict(X_test)
    accuracy = accuracy_score(y_test, preds)
    return accuracy

In [7]:
# Create Optuna study for optimization with persistent storage
study = optuna.create_study(
    direction='maximize',  # Assuming you are maximizing accuracy
    study_name="XGboost_diseases_symptoms_dropextremelymore500withSMOTE_study", 
    storage=r"sqlite:///C:/Users/khiew/Downloads/xgboost.db", 
    load_if_exists=True  # Load the study if it already exists, to resume from the last trial
)

# Optimize the study with your objective function, you can adjust the n_trials as needed
study.optimize(objective, n_trials=20)
# Print the best trial and hyperparameters
print("\nBest Trial:")
print(study.best_trial)
print("Best Hyperparameters:")
print(study.best_trial.params)

[I 2025-04-25 02:19:58,345] Using an existing study with name 'XGboost_diseases_symptoms_dropextremelymore500withSMOTE_study' instead of creating a new one.
[I 2025-04-25 02:24:08,685] Trial 1 finished with value: 0.39869436201780417 and parameters: {'max_depth': 5, 'learning_rate': 0.07732283089992464, 'n_estimators': 393, 'subsample': 0.6770099386672526, 'colsample_bytree': 0.7529758343608222, 'gamma': 3.615640288513469, 'reg_alpha': 4.987950106469795, 'reg_lambda': 0.05404270871717132}. Best is trial 1 with value: 0.39869436201780417.
[I 2025-04-25 02:29:16,159] Trial 2 finished with value: 0.3958160237388724 and parameters: {'max_depth': 6, 'learning_rate': 0.2775821253882215, 'n_estimators': 537, 'subsample': 0.9412861266158414, 'colsample_bytree': 0.8301582023010219, 'gamma': 3.8363379868710505, 'reg_alpha': 0.9411204765774939, 'reg_lambda': 1.020397071120872}. Best is trial 1 with value: 0.39869436201780417.
[I 2025-04-25 02:32:36,605] Trial 3 finished with value: 0.399050445103


Best Trial:
FrozenTrial(number=5, state=TrialState.COMPLETE, values=[0.40445103857566767], datetime_start=datetime.datetime(2025, 4, 25, 2, 34, 33, 441832), datetime_complete=datetime.datetime(2025, 4, 25, 2, 42, 14, 215), params={'max_depth': 11, 'learning_rate': 0.12187124127084109, 'n_estimators': 806, 'subsample': 0.9574049069856616, 'colsample_bytree': 0.7149187586980786, 'gamma': 0.7449897920404186, 'reg_alpha': 2.1273951947909904, 'reg_lambda': 3.8972064490298552}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'max_depth': IntDistribution(high=15, log=False, low=3, step=1), 'learning_rate': FloatDistribution(high=0.3, log=False, low=0.01, step=None), 'n_estimators': IntDistribution(high=1000, log=False, low=100, step=1), 'subsample': FloatDistribution(high=1.0, log=False, low=0.5, step=None), 'colsample_bytree': FloatDistribution(high=1.0, log=False, low=0.5, step=None), 'gamma': FloatDistribution(high=5.0, log=False, low=0.0, step=None), 'reg_alpha': F